# Vietnamese Embedding Models — Document Retrieval Evaluation

## ⚠️ Why Not ViSim-400 for Retrieval?

**ViSim-400 is a semantic similarity (STS) dataset** — it contains sentence *pairs* with human similarity scores (0–4). It does not have:
- A corpus of documents to retrieve from
- Natural language queries
- Relevance judgments mapping queries → relevant documents

These three components are required for a proper retrieval evaluation (the BEIR/MTEB standard format).

**This notebook uses proper Vietnamese retrieval datasets instead**, following the same MTEB format. We cover two approaches:

| Approach | When to use |
|---|---|
| **A) MTEB library (recommended)** | Standard benchmark comparison, reproducible scores |
| **B) Custom retrieval pipeline** | Your own corpus / full control over metrics |

## Retrieval Metrics Used

| Metric | What it measures |
|---|---|
| **nDCG@10** | Ranking quality of top-10 results (main MTEB metric) |
| **MAP@10** | Mean Average Precision — rewards early relevant hits |
| **Recall@k** | Fraction of relevant docs found in top-k |
| **MRR@10** | Mean Reciprocal Rank — rewards the first relevant hit |

---
# Part A: MTEB-Based Evaluation (Recommended)

Uses official Vietnamese retrieval datasets from the MTEB benchmark:
- **VieQuAD**: Vietnamese SQuAD — question-answer retrieval
- **ZaloE2ERetrieval**: Zalo AI 2021 legal text retrieval
- **HotpotQA-VN**: Vietnamese HotpotQA multi-hop QA retrieval (from VN-MTEB)

These datasets cover different domains and retrieval scenarios. The main metric from MTEB is nDCG@10 (Normalized Discounted Cumulative Gain at 10), which measures how well the model retrieves relevant results and ranks them.

## A.1 Install Dependencies

In [ ]:
!pip install mteb sentence-transformers datasets pandas matplotlib seaborn pyvi -q

# Verify mteb version
import mteb
print(f"MTEB version: {mteb.__version__}")

## A.2 Define Models to Evaluate

In [ ]:
# Models to benchmark
# Each string is a HuggingFace model ID that mteb.get_model() can load
MODEL_IDS = [
    "AITeamVN/Vietnamese_Embedding",          # BGE-M3 fine-tuned on 300K Vietnamese triplets
    "dangvantuan/vietnamese-embedding",        # PhoBERT-based, 4-stage training
    "dangvantuan/vietnamese-document-embedding", # GTE-multilingual, 8096-token context
    "keepitreal/vietnamese-sbert",             # PhoBERT baseline
    "BAAI/bge-m3",                            # Multilingual baseline
]

# Friendly labels for charts
MODEL_LABELS = {
    "AITeamVN/Vietnamese_Embedding":              "AITeamVN (BGE-M3)",
    "dangvantuan/vietnamese-embedding":            "VN-Embed (PhoBERT)",
    "dangvantuan/vietnamese-document-embedding":   "VN-DocEmbed (GTE)",
    "keepitreal/vietnamese-sbert":                 "VN-SBERT",
    "BAAI/bge-m3":                                "BGE-M3 (multilingual)",
}

print(f"Will evaluate {len(MODEL_IDS)} models.")

## A.3 Select Retrieval Tasks

In [ ]:
# Vietnamese retrieval tasks available in MTEB
# See full list: mteb.get_tasks(task_types=["Retrieval"], languages=["vie"])

TASK_NAMES = [
    "VieQuAD",           # Vietnamese QA retrieval (SQuAD-style)
    "ZaloE2ERetrieval",  # Zalo AI legal text retrieval
    "HotpotQA-VN",       # Multi-hop QA retrieval (VN-MTEB)
]

# Preview the tasks
tasks = mteb.get_tasks(tasks=TASK_NAMES)
for t in tasks:
    print(f"  Task: {t.metadata.name} | Type: {t.metadata.type} | Languages: {t.metadata.languages}")

## A.4 Run Evaluation

In [ ]:
import os
import json
import pandas as pd

OUTPUT_DIR = "./mteb_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_results = {}  # model_id -> {task_name -> score}

for model_id in MODEL_IDS:
    print(f"\n{'='*60}")
    print(f"Evaluating: {MODEL_LABELS[model_id]}")
    print(f"{'='*60}")

    model = mteb.get_model(model_id)
    evaluation = mteb.MTEB(tasks=mteb.get_tasks(tasks=TASK_NAMES))

    results = evaluation.run(
        model,
        output_folder=os.path.join(OUTPUT_DIR, model_id.replace("/", "__")),
        eval_splits=["test"],
        overwrite_results=False,   # set True to re-run even if cached
    )

    # Extract nDCG@10 for each task
    model_scores = {}
    for result in results:
        task_name = result.task_name
        # nDCG@10 is the primary MTEB retrieval metric
        ndcg = result.scores["test"][0].get("ndcg_at_10", None)
        model_scores[task_name] = round(ndcg * 100, 2) if ndcg else None
        print(f"  {task_name}: nDCG@10 = {model_scores[task_name]}")

    all_results[model_id] = model_scores

print("\n✅ All evaluations complete.")

## A.5 Results Table & Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Build summary DataFrame
rows = []
for model_id, scores in all_results.items():
    row = {"Model": MODEL_LABELS[model_id]}
    for task in TASK_NAMES:
        row[task] = scores.get(task)
    row["Average nDCG@10"] = round(
        np.nanmean([v for v in scores.values() if v is not None]), 2
    )
    rows.append(row)

summary = pd.DataFrame(rows).sort_values("Average nDCG@10", ascending=False).reset_index(drop=True)
summary.index += 1
summary.index.name = "Rank"

print("=== Vietnamese Retrieval Benchmark — nDCG@10 (higher = better) ===")
print(summary.to_string())
summary

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.05)

# --- Grouped bar chart ---
fig, ax = plt.subplots(figsize=(12, 6))

models = summary["Model"].tolist()
x = np.arange(len(models))
width = 0.2
colors = ["#4C9BE8", "#F4845F", "#6BCB77", "#FFD166"]

for i, task in enumerate(TASK_NAMES):
    vals = summary[task].tolist()
    bars = ax.bar(x + i * width, vals, width, label=task, color=colors[i % len(colors)])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x + width * (len(TASK_NAMES) - 1) / 2)
ax.set_xticklabels(models, rotation=15, ha="right")
ax.set_ylabel("nDCG@10 (%)")
ax.set_title("Vietnamese Document Retrieval — nDCG@10 per Task")
ax.legend(title="Task")
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig("retrieval_comparison.png", dpi=150)
plt.show()

In [ ]:
# --- Heatmap ---
heat_data = summary.set_index("Model")[TASK_NAMES + ["Average nDCG@10"]]

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(
    heat_data.astype(float),
    annot=True, fmt=".1f", cmap="YlGnBu",
    linewidths=0.5, ax=ax, cbar_kws={"label": "nDCG@10 (%)"}
)
ax.set_title("Retrieval Performance Heatmap")
plt.tight_layout()
plt.savefig("retrieval_heatmap.png", dpi=150)
plt.show()

---
# Part B: Custom Retrieval Pipeline (Full Control)

Use this when:
- You have your **own Vietnamese corpus** (e.g., company documents, news articles)
- You want to compute **multiple metrics**: nDCG@k, MAP@k, Recall@k, MRR@k
- You want to inspect **per-query** results and do error analysis

We simulate a retrieval setup using ViSim-400 by converting high-similarity pairs into
pseudo query→document relevance judgments — this is a common technique when no
dedicated retrieval dataset is available.

## B.1 Build a Retrieval Dataset from ViSim-400

We treat `sentence_1` as the **query** and all `sentence_2` values as the **corpus**.
A document is considered *relevant* if its similarity score with the query exceeds a threshold.

In [ ]:
!pip install seacrowd -q

from datasets import load_dataset
import pandas as pd

dataset = load_dataset("SEACrowd/visim400", trust_remote_code=True)
split = list(dataset.keys())[0]
df = dataset[split].to_pandas()

print("Columns:", df.columns.tolist())
print(f"Rows: {len(df)}")
df.head(3)

In [ ]:
# ---- Adjust column names to match the dataset ----
COL_SENT1 = "text_1"   # query
COL_SENT2 = "text_2"   # document
COL_SCORE = "label"    # human similarity score

# Normalize scores to [0, 1]
df["score_norm"] = df[COL_SCORE].astype(float)
if df["score_norm"].max() > 1.0:
    df["score_norm"] = df["score_norm"] / df["score_norm"].max()

# Threshold for relevance: pairs with score >= threshold are 'relevant'
RELEVANCE_THRESHOLD = 0.6

queries  = df[COL_SENT1].tolist()   # each sentence_1 is a query
corpus   = df[COL_SENT2].tolist()   # all sentence_2 form the corpus

# qrels[query_idx] = set of corpus_idx that are relevant to this query
qrels = {}
for i, row in df.iterrows():
    if row["score_norm"] >= RELEVANCE_THRESHOLD:
        qrels.setdefault(i, set()).add(i)  # sentence_2[i] is relevant to query[i]

print(f"Queries: {len(queries)}")
print(f"Corpus size: {len(corpus)}")
print(f"Queries with at least one relevant doc (threshold={RELEVANCE_THRESHOLD}): {len(qrels)}")

## B.2 Retrieval Metric Functions

In [ ]:
import numpy as np

def dcg_at_k(ranked_relevance, k):
    """Compute DCG@k."""
    ranked_relevance = ranked_relevance[:k]
    gains = [rel / np.log2(rank + 2) for rank, rel in enumerate(ranked_relevance)]
    return sum(gains)

def ndcg_at_k(ranked_relevance, k, n_relevant):
    """Compute nDCG@k."""
    ideal = [1] * min(n_relevant, k) + [0] * max(0, k - n_relevant)
    idcg = dcg_at_k(ideal, k)
    if idcg == 0:
        return 0.0
    return dcg_at_k(ranked_relevance, k) / idcg

def average_precision_at_k(ranked_relevance, k, n_relevant):
    """Compute AP@k."""
    if n_relevant == 0:
        return 0.0
    hits, total = 0, 0
    for rank, rel in enumerate(ranked_relevance[:k]):
        if rel:
            hits += 1
            total += hits / (rank + 1)
    return total / min(n_relevant, k)

def recall_at_k(ranked_relevance, k, n_relevant):
    """Compute Recall@k."""
    if n_relevant == 0:
        return 0.0
    return sum(ranked_relevance[:k]) / n_relevant

def mrr_at_k(ranked_relevance, k):
    """Compute MRR@k."""
    for rank, rel in enumerate(ranked_relevance[:k]):
        if rel:
            return 1.0 / (rank + 1)
    return 0.0

def compute_all_metrics(ranked_relevance, k, n_relevant):
    return {
        f"nDCG@{k}":   ndcg_at_k(ranked_relevance, k, n_relevant),
        f"MAP@{k}":    average_precision_at_k(ranked_relevance, k, n_relevant),
        f"Recall@{k}": recall_at_k(ranked_relevance, k, n_relevant),
        f"MRR@{k}":   mrr_at_k(ranked_relevance, k),
    }

print("Metric functions defined.")

## B.3 Run Custom Retrieval Evaluation

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import torch

try:
    from pyvi.ViTokenizer import tokenize as vi_tokenize
except ImportError:
    vi_tokenize = None

CUSTOM_MODELS = [
    {"name": "dangvantuan/vietnamese-embedding",          "label": "VN-Embed (PhoBERT)",   "needs_pyvi": True},
    {"name": "AITeamVN/Vietnamese_Embedding",             "label": "AITeamVN (BGE-M3)",    "needs_pyvi": False},
    {"name": "dangvantuan/vietnamese-document-embedding", "label": "VN-DocEmbed (GTE)",    "needs_pyvi": False, "trust_remote_code": True},
    {"name": "keepitreal/vietnamese-sbert",               "label": "VN-SBERT",             "needs_pyvi": False},
    {"name": "BAAI/bge-m3",                              "label": "BGE-M3 (multilingual)", "needs_pyvi": False},
]

K_VALUES = [1, 5, 10]  # evaluate at these cutoffs

custom_results = []

for cfg in CUSTOM_MODELS:
    print(f"\n{'='*60}")
    print(f"Evaluating: {cfg['label']}")
    print(f"{'='*60}")

    if cfg.get("needs_pyvi") and vi_tokenize is None:
        print("  Skipping: pyvi not installed")
        continue

    # Apply Vietnamese tokenization if required
    q_input = [vi_tokenize(s) for s in queries] if cfg.get("needs_pyvi") else queries
    c_input = [vi_tokenize(s) for s in corpus]  if cfg.get("needs_pyvi") else corpus

    trust = cfg.get("trust_remote_code", False)
    model = SentenceTransformer(cfg["name"], trust_remote_code=trust)

    print("  Encoding corpus...")
    corpus_emb = model.encode(c_input, batch_size=64, show_progress_bar=True,
                               convert_to_tensor=True, normalize_embeddings=True)

    print("  Encoding queries...")
    query_emb = model.encode(q_input, batch_size=64, show_progress_bar=True,
                              convert_to_tensor=True, normalize_embeddings=True)

    # Compute full similarity matrix: [n_queries, n_corpus]
    sim_matrix = cos_sim(query_emb, corpus_emb).cpu().numpy()

    # Evaluate only queries that have at least one relevant document
    metric_sums = {f"{m}@{k}": 0.0
                   for m in ["nDCG", "MAP", "Recall", "MRR"]
                   for k in K_VALUES}
    n_evaluated = 0

    for q_idx, relevant_docs in qrels.items():
        sims = sim_matrix[q_idx]  # similarities of this query vs all corpus docs
        ranked_indices = np.argsort(sims)[::-1]  # descending order

        for k in K_VALUES:
            ranked_rel = [1 if idx in relevant_docs else 0 for idx in ranked_indices[:k]]
            metrics = compute_all_metrics(ranked_rel, k, len(relevant_docs))
            for metric_name, val in metrics.items():
                metric_sums[metric_name] += val

        n_evaluated += 1

    # Average over all evaluated queries
    avg_metrics = {k: round(v / n_evaluated, 4) for k, v in metric_sums.items()}
    avg_metrics["Model"] = cfg["label"]

    print(f"  Evaluated on {n_evaluated} queries")
    for metric, val in avg_metrics.items():
        if metric != "Model":
            print(f"    {metric}: {val:.4f}")

    custom_results.append(avg_metrics)

print("\n✅ Custom retrieval evaluation complete.")

## B.4 Results Summary

In [ ]:
# Build results table
metric_cols = [f"{m}@{k}" for m in ["nDCG", "MAP", "Recall", "MRR"] for k in K_VALUES]
custom_df = pd.DataFrame(custom_results)[["Model"] + metric_cols]
custom_df = custom_df.sort_values("nDCG@10", ascending=False).reset_index(drop=True)
custom_df.index += 1
custom_df.index.name = "Rank"

print("=== Custom Retrieval Results (ViSim-400 pseudo-retrieval) ===")
custom_df

In [ ]:
# --- Multi-metric bar chart at k=10 ---
k = 10
metric_names = [f"nDCG@{k}", f"MAP@{k}", f"Recall@{k}", f"MRR@{k}"]
colors = ["#4C9BE8", "#F4845F", "#6BCB77", "#FFD166"]

fig, ax = plt.subplots(figsize=(12, 5))
models = custom_df["Model"].tolist()
x = np.arange(len(models))
width = 0.18

for i, metric in enumerate(metric_names):
    vals = custom_df[metric].tolist()
    bars = ax.bar(x + i * width, vals, width, label=metric, color=colors[i])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(models, rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.1)
ax.set_title(f"Custom Retrieval Evaluation @{k} — ViSim-400 Pseudo-Retrieval")
ax.legend()
plt.tight_layout()
plt.savefig("custom_retrieval_metrics.png", dpi=150)
plt.show()

## B.5 Error Analysis — Per-Query Breakdown

In [ ]:
# Inspect which queries the BEST model struggled with most
# (Re-run with the top model for detailed per-query analysis)

best_model_cfg = CUSTOM_MODELS[0]  # change index to the best model from B.4

if best_model_cfg.get("needs_pyvi") and vi_tokenize:
    q_input = [vi_tokenize(s) for s in queries]
    c_input = [vi_tokenize(s) for s in corpus]
else:
    q_input, c_input = queries, corpus

model = SentenceTransformer(best_model_cfg["name"],
                             trust_remote_code=best_model_cfg.get("trust_remote_code", False))
corpus_emb = model.encode(c_input, batch_size=64, convert_to_tensor=True, normalize_embeddings=True)
query_emb  = model.encode(q_input, batch_size=64, convert_to_tensor=True, normalize_embeddings=True)
sim_matrix = cos_sim(query_emb, corpus_emb).cpu().numpy()

per_query = []
for q_idx, relevant_docs in qrels.items():
    ranked = np.argsort(sim_matrix[q_idx])[::-1]
    ranked_rel = [1 if idx in relevant_docs else 0 for idx in ranked[:10]]
    per_query.append({
        "query": queries[q_idx],
        "relevant_doc": corpus[list(relevant_docs)[0]],
        "rank_of_relevant": next((r + 1 for r, v in enumerate(ranked_rel) if v), ">10"),
        "nDCG@10": ndcg_at_k(ranked_rel, 10, len(relevant_docs)),
        "top1_retrieved": corpus[ranked[0]],
    })

per_query_df = pd.DataFrame(per_query)

print("Queries where model FAILED (relevant doc not in top 10):")
failed = per_query_df[per_query_df["rank_of_relevant"] == ">10"]
failed[["query", "relevant_doc", "top1_retrieved"]].head(10)

In [ ]:
# Distribution of rank of first relevant document
rank_numeric = per_query_df["rank_of_relevant"].apply(
    lambda x: 11 if x == ">10" else int(x)
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rank_numeric, bins=range(1, 13), edgecolor="white", color="#4C9BE8", align="left")
ax.set_xticks(range(1, 12))
ax.set_xticklabels([str(i) for i in range(1, 11)] + [">10"])
ax.set_xlabel("Rank of First Relevant Document")
ax.set_ylabel("Number of Queries")
ax.set_title(f"Rank Distribution — {best_model_cfg['label']}")
plt.tight_layout()
plt.savefig("rank_distribution.png", dpi=150)
plt.show()

## B.6 Save All Results

In [ ]:
custom_df.to_csv("retrieval_results.csv")
per_query_df.to_csv("per_query_analysis.csv", index=False)
print("Results saved: retrieval_results.csv, per_query_analysis.csv")

---
## Summary: STS vs Retrieval Evaluation

| Aspect | STS (Part 1 notebook) | Retrieval (This notebook) |
|---|---|---|
| **Input** | Sentence pairs + similarity scores | Queries + corpus + relevance judgments |
| **Task** | How similar are these two sentences? | Given a query, rank all documents |
| **Metric** | Pearson / Spearman correlation | nDCG@k, MAP@k, Recall@k, MRR@k |
| **Dataset** | ViSim-400 | VieQuAD, ZaloE2E, HotpotQA-VN |
| **Real-world use** | Duplicate detection, paraphrase | Search engines, RAG retrieval |

**Recommendation**: Use **Part A (MTEB)** for standard benchmarking and comparison against published results. Use **Part B (custom)** when you have a domain-specific corpus or need detailed metric breakdowns.